In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_squared_error, r2_score
from pandas.core.arrays import categorical
from sklearn.metrics import classification_report
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor

In [ ]:
from google.colab import files
uploaded = files.upload()
import zipfile
with zipfile.ZipFile("house-prices-advanced-regression-techniques.zip","r")as zip_ref:
  zip_ref.extractall()

Saving house-prices-advanced-regression-techniques.zip to house-prices-advanced-regression-techniques.zip


In [ ]:
train=pd.read_csv("train.csv")
test=pd.read_csv("test.csv")

In [ ]:
print(train.describe())
print(train.info())
print(train.shape)
print(train.columns)
print(train.head())
train.isna().sum()

                Id   MSSubClass  LotFrontage        LotArea  OverallQual  \
count  1460.000000  1460.000000  1201.000000    1460.000000  1460.000000   
mean    730.500000    56.897260    70.049958   10516.828082     6.099315   
std     421.610009    42.300571    24.284752    9981.264932     1.382997   
min       1.000000    20.000000    21.000000    1300.000000     1.000000   
25%     365.750000    20.000000    59.000000    7553.500000     5.000000   
50%     730.500000    50.000000    69.000000    9478.500000     6.000000   
75%    1095.250000    70.000000    80.000000   11601.500000     7.000000   
max    1460.000000   190.000000   313.000000  215245.000000    10.000000   

       OverallCond    YearBuilt  YearRemodAdd   MasVnrArea   BsmtFinSF1  ...  \
count  1460.000000  1460.000000   1460.000000  1452.000000  1460.000000  ...   
mean      5.575342  1971.267808   1984.865753   103.685262   443.639726  ...   
std       1.112799    30.202904     20.645407   181.066207   456.098091  ..

,0
Id,0
MSSubClass,0
MSZoning,0
LotFrontage,259
LotArea,0
...,...
MoSold,0
YrSold,0
SaleType,0
SaleCondition,0


In [ ]:
X_train=train[["LotArea","Utilities","OverallQual","TotRmsAbvGrd","SaleCondition","MiscVal","CentralAir"]].copy()
Y_train=np.log1p(train["SalePrice"])
X_test=test[["LotArea","Utilities","OverallQual","TotRmsAbvGrd","SaleCondition","MiscVal","CentralAir"]].copy()

In [ ]:
numeric_features=["LotArea","OverallQual","TotRmsAbvGrd","MiscVal"]
categorical_features=["Utilities","SaleCondition","CentralAir"]
numeric_transformer=Pipeline(steps=[
    ("imputer",SimpleImputer(strategy="mean")),
    ('scaler', StandardScaler())])
categorical_transformer=Pipeline(steps=[
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])
preprocessing = ColumnTransformer(
     transformers=[("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)])

In [ ]:
models={
    "Linear": LinearRegression(),
    "RF": RandomForestRegressor(),
    "GB": GradientBoostingRegressor(),
    "XGB":XGBRegressor()
    }
param_grid={
    "Linear": {},
    "RF": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 10] },
    "GB": {
        "model__n_estimators": [100, 200],
        "model__learning_rate": [0.05, 0.1]},
    "XGB": {
        "model__n_estimators": [100, 200],
        "model__learning_rate": [0.05, 0.1],
        "model__max_depth": [None, 10]}
        }

In [ ]:
best_models= {}
for name,model in models.items():
    pipeline = Pipeline(steps=[
      ("preprocessing", preprocessing),
      ("model", model)])
    grid=GridSearchCV(
      estimator=pipeline ,
       param_grid= param_grid[name] ,
       cv=5 , scoring="neg_root_mean_squared_error" )
    grid.fit(X_train,Y_train)
    print(name,-grid.best_score_)
    best_models[name] = grid.best_estimator_
best_model = best_models["XGB"]

Linear 0.19863352358423852
RF 0.1951425358013966
GB 0.18804588732922328
XGB 0.19501176305097817


In [ ]:
pred=best_model.predict(X_test)
pred = np.expm1(pred)
print(pred[:10])

[136228.44177777 168652.92157038 149108.3680036  171230.71014365
 224668.95158801 171230.71014365 148441.46533827 162805.15925551
 192636.74781583 108852.29416076]


In [ ]:
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": pred })
submission.to_csv("submission.csv", index=False)
from google.colab import files
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>